https://zenodo.org/records/10695923?preview_file=carotid_bifurcation_database.zip

this is where the data is obtained from

In [ ]:
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyvista as pv
import pyvistaqt as pvqt
import scipy.ndimage as spy_ndimage
import scipy.spatial as spy_spatial
import scipy.spatial.transform as spy_transform

In [ ]:
case_dir, side = "k_008", "right"
centerlines = pv.read(
  f"./crops/data/carotid/{case_dir}/{case_dir}_{side}_lumen_centerlines.vtp"
)
radius = np.asarray(centerlines["MaximumInscribedSphereRadius"])

In [ ]:
%matplotlib qt5


# Every polyline is stored as [num_points, id_0, ..., id_{num_points - 1}].
def plot_centerlines(
  centerlines,
  highlight_segments=(),
  lw=1.0,
  highlight_lw=3.0,
  dim_alpha=0.5,
  figsize=(4.0, 6.0),
  verbose=True,
):
  lines, cursor = [], 0
  while cursor < centerlines.lines.size:
    num_points = centerlines.lines[cursor]
    lines.append(centerlines.lines[cursor + 1 : cursor + 1 + num_points])
    cursor += num_points + 1

  highlight = set(highlight_segments)
  ax = plt.figure(figsize=figsize).add_subplot(projection="3d")
  for index, point_ids in enumerate(lines):
    points = centerlines.points[point_ids]
    length = np.linalg.norm(np.diff(points, axis=0), axis=-1).sum()
    if verbose:
      print(f"line {index}  length {length:.1f} mm")
    is_hl = index in highlight
    alpha = 1.0 if (is_hl or not highlight) else dim_alpha
    (line,) = ax.plot(
      points[:, 0],
      points[:, 1],
      points[:, 2],
      lw=highlight_lw if is_hl else lw,
      alpha=alpha,
      label=f"{index}",
    )
    mid = points[len(points) // 2]
    ax.text(
      mid[0],
      mid[1],
      mid[2],
      f"{index}",
      fontsize=8,
      ha="center",
      va="center",
      color=line.get_color(),
      alpha=alpha,
      fontweight="bold" if is_hl else "normal",
    )

  ax.set_aspect("equal")
  ax.set_axis_off()
  ax.legend(frameon=False)
  plt.show()
  return lines, ax


lines, ax = plot_centerlines(centerlines, highlight_segments=[0, 4, 5])

### Extract stent (line no and param coord)

In [ ]:
line_index, t_start, t_end = 4, 0.75, 1.00
spacing, smoothing = 0.25, 2.0  # mm
camera_position = None  # lumen view, set from a saved header to redo its figure

In [ ]:
point_ids = lines[line_index]
raw_points = centerlines.points[point_ids].astype(float)
raw_arc = np.concatenate(
  [[0.0], np.cumsum(np.linalg.norm(np.diff(raw_points, axis=0), axis=-1))]
)
# Resample uniformly and smooth the whole line over a few mm, since the centerline
# carries sub-millimetre jitter that would otherwise dominate the curvature.
arc = np.arange(0.0, raw_arc[-1], spacing)
points = np.stack(
  [np.interp(arc, raw_arc, raw_points[:, dim]) for dim in range(3)], axis=-1
)
points = spy_ndimage.gaussian_filter1d(
  points, sigma=smoothing / spacing, axis=0, mode="nearest"
)

In [ ]:
line_radius = np.interp(arc, raw_arc, radius[point_ids])
chords = np.diff(points, axis=0)
chords /= np.linalg.norm(chords, axis=-1, keepdims=True)

start, stop = np.searchsorted(arc / arc[-1], [t_start, t_end])
segment = points[start:stop]

### Plot and see the segment

In [ ]:
context_color, section_color, context_opacity = "firebrick", "orange", 0.15
lumen = pv.read(f"./crops/data/carotid/{case_dir}/{case_dir}_{side}_lumen.stl")

# Each lumen cell is assigned to its nearest point on the chosen smoothed centerline.
# It counts as part of the section when that point lies inside the window and the
# cell is within 1.5 lumen radii of it, which keeps walls of neighbouring branches out.
distance, nearest = spy_spatial.cKDTree(points).query(lumen.cell_centers().points)
in_section = (
  (nearest >= start) & (nearest < stop) & (distance < 1.5 * line_radius[nearest])
)

# Drawn exactly as in carotid_reader_catheter.ipynb, so the paper figures match.
lumen_plotter = pvqt.BackgroundPlotter(window_size=(1200, 900))
lumen_plotter.enable_depth_peeling()
lumen_plotter.add_mesh(
  lumen.extract_cells(np.flatnonzero(~in_section)),
  color=context_color,
  opacity=context_opacity,
  smooth_shading=True,
)
lumen_plotter.add_mesh(
  lumen.extract_cells(np.flatnonzero(in_section)),
  color=section_color,
  smooth_shading=True,
)
lumen_plotter.set_background("white")
if camera_position is not None:
  lumen_plotter.camera_position = camera_position

### Metrics

In [ ]:
turn = np.arccos(np.clip(np.einsum("nd,nd->n", chords[:-1], chords[1:]), -1.0, 1.0))
segment_turn = turn[start : stop - 2]

length = spacing * (segment.shape[0] - 1)
mean_radius = float(np.mean(line_radius[start:stop]))
aspect_ratio = length / mean_radius
tortuosity = length / np.linalg.norm(segment[-1] - segment[0])
total_turn = np.degrees(segment_turn.sum())
tightest_bend = spacing / segment_turn.max() / mean_radius
narrowing = np.min(line_radius[start:stop]) / mean_radius

print(
  f"length {length:.1f} mm,  mean radius {mean_radius:.2f} mm,  L/R {aspect_ratio:.1f}"
)
print(f"tortuosity L/chord {tortuosity:.2f},  total turning {total_turn:.0f} deg")
print(f"tightest bend {tightest_bend:.2f} R,  min/mean radius {narrowing:.2f}")


### Check if the segment is a fit for a "good stent"

In [ ]:
# Tightest bend the stent can follow, in stent radii, for dEps = 0.454 (thin tube).
reach_bend_ratio = 3.4
stent_length_range = (20.0, 60.0)  # clinical carotid stents, mm

checks = {
  f"length outside clinical carotid stents {stent_length_range} mm": not (
    stent_length_range[0] <= length <= stent_length_range[1]
  ),
  "L/R outside 5 to 30": not 5.0 <= aspect_ratio <= 50.0,
  "nearly straight, under 30 deg of turning": total_turn < 30.0,
  # "severe narrowing in the window, so the mean radius is biased": narrowing < 0.5,
}
problems = [message for message, failed in checks.items() if failed]
for message in problems:
  print(f"WARNING  {message}")
if problems:
  print("NOT A GOOD TARGET")
elif tightest_bend >= reach_bend_ratio:
  print("GOOD TARGET, within reach")
else:
  print("GOOD TARGET, beyond reach at its tightest bend (use as the limit case)")


In [ ]:
fig = plt.figure(figsize=(8.0, 4.0))
path_ax = fig.add_subplot(1, 2, 1, projection="3d")
path_ax.plot(*points.T, color="0.7", lw=1.0)
path_ax.plot(*segment.T, color="k", lw=2.0)
for t_mark in np.linspace(0.0, 1.0, 11):
  path_ax.text(
    *points[int(t_mark * (points.shape[0] - 1))], f"{t_mark:.1f}", fontsize=7
  )
path_ax.set_aspect("equal")
path_ax.set_axis_off()
curvature_ax = fig.add_subplot(1, 2, 2)
curvature_ax.axvspan(t_start, t_end, color="0.9")
curvature_ax.plot(arc[1:-1] / arc[-1], turn / spacing * mean_radius, color="k", lw=1.0)
curvature_ax.axhline(1.0 / reach_bend_ratio, color="0.5", ls="--", lw=0.8)
curvature_ax.set_xlabel(r"$t$")
curvature_ax.set_ylabel(r"$|\kappa|\,\bar{R}$")
fig.tight_layout()
plt.show()

### Save the segment and the figure

In [ ]:
# Start at the origin with the initial tangent along +x, the dry stent axis at the
# clamp, and save in m like the NU target.
segment = segment - segment[0]
rotation, _ = spy_transform.Rotation.align_vectors([[1.0, 0.0, 0.0]], [segment[1]])
segment = rotation.apply(segment)

save_in_m = 1.0e-3
camera_position = np.asarray(lumen_plotter.camera_position.to_list()).tolist()

# The header repeats the picks exactly as they are typed in the cells above, so
# pasting its lines back there regenerates this file and the figure. Read it with
# pd.read_csv(fname, comment="#").
fname = f"crops/data/stent_centerlines/{case_dir}_{side}_{line_index}.csv"
header = (
  "# made by crops/notebooks/dev/etc/carotid_reader_stent.ipynb\n"
  f'# case_dir, side = "{case_dir}", "{side}"\n'
  f"# line_index, t_start, t_end = {line_index}, {t_start}, {t_end}\n"
  f"# spacing, smoothing = {spacing}, {smoothing}  # mm\n"
  f"# camera_position = {camera_position}\n"
)
with open(fname, "w", newline="") as file:
  file.write(header)
  pd.DataFrame(
    save_in_m
    * np.column_stack(
      [spacing * np.arange(segment.shape[0]), segment, line_radius[start:stop]]
    ),
    columns=["s", "x", "y", "z", "r"],
  ).to_csv(file, index=False)

# The lumen view as set by hand in its Qt window, at 4x the window size and cropped
# to the vessel.
fig_name = pathlib.Path(f"results/paper/stent/{case_dir}_{side}_{line_index}/carotid.png")
fig_name.parent.mkdir(parents=True, exist_ok=True)
image = lumen_plotter.screenshot(return_img=True, scale=4)
foreground = np.any(image[..., :3] < 255, axis=-1)
rows = np.flatnonzero(foreground.any(axis=1))
cols = np.flatnonzero(foreground.any(axis=0))
plt.imsave(fig_name, image[rows[0] : rows[-1] + 1, cols[0] : cols[-1] + 1])

print(f"saved {fname} and {fig_name}")
print(header, end="")